# Parsing evaluation

Declaring the paths and the two final parse configs (SVM, Neural Nets) plus the loader that pairs each golden slide with its parsed chunk, this notebook scores the production parse, not A/B

In [ ]:
import json, re, unicodedata
from pathlib import Path
import pandas as pd
import os
import re
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

ROOT     = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PARSED   = ROOT / "data/parsed_NO_EDIT/machine_learning"
GOLDEN   = ROOT / "data/eval/test_jsonfiles/golden"
EVAL_OUT = ROOT / "data/eval"
EVAL_OUT.mkdir(parents=True, exist_ok=True)

CONFIGS = [
    {"vorlesung": "SVM",
     "parse":  PARSED / "ML_5_svm/ML_5_svm_chunks.json",
     "golden": GOLDEN / "ML_5_svm_golden_parse.json"},
    {"vorlesung": "Neuronale Netze",
     "parse":  PARSED / "ML_9_neuronale_netze/ML_9_neuronale_netze_chunks.json",
     "golden": GOLDEN / "ML_9_neuronale_netze_golden_parse.json"},
]

def load_pairs(cfg):
    parse  = json.loads(Path(cfg["parse"]).read_text(encoding="utf-8"))
    golden = json.loads(Path(cfg["golden"]).read_text(encoding="utf-8"))
    by_id  = {c["id"]: c for c in parse}
    pairs  = [(g, by_id[g["slide_id"]]) for g in golden if g["slide_id"] in by_id]
    return golden, by_id, pairs

for cfg in CONFIGS:
    golden, by_id, pairs = load_pairs(cfg)
    print(f"{cfg['vorlesung']:16s} | golden={len(golden):3d}  pairs={len(pairs):3d}")

## Normalisation

Defining normalize (LaTeX to Unicode, case and punctuation folding) so string based recall does not care about surface formatting

In [ ]:
LATEX = {
    r"\alpha": "α", r"\beta": "β", r"\gamma": "γ", r"\delta": "δ",
    r"\epsilon": "ε", r"\varepsilon": "ε", r"\zeta": "ζ", r"\eta": "η",
    r"\theta": "θ", r"\kappa": "κ", r"\lambda": "λ", r"\mu": "μ",
    r"\nu": "ν", r"\xi": "ξ", r"\pi": "π", r"\rho": "ρ", r"\sigma": "σ",
    r"\tau": "τ", r"\phi": "φ", r"\chi": "χ", r"\psi": "ψ", r"\omega": "ω",
    r"\leq": "≤", r"\le": "≤", r"\geq": "≥", r"\ge": "≥",
    r"\neq": "≠", r"\ne": "≠", r"\approx": "≈", r"\times": "×",
    r"\pm": "±", r"\infty": "∞", r"\sum": "∑", r"\partial": "∂",
    r"\nabla": "∇", r"\in": "∈", r"\rightarrow": "→", r"\to": "→",
}

def normalize(t: str) -> str:
    t = unicodedata.normalize("NFC", t)
    t = t.lower()
    for cmd in sorted(LATEX, key=len, reverse=True):    
        t = re.sub(re.escape(cmd) + r"(?![a-z])", LATEX[cmd], t)
    t = t.replace(",,", "").replace("``", "").replace("''", "")  
    t = re.sub(r'[„“”‚‘’»«"]', "", t)                    
    t = re.sub(r"\$+", " ", t)                           
    t = re.sub(r"[*#`>~]", " ", t)                   
    t = re.sub(r"…", "...", t)                          
    t = re.sub(r"[‐-―−]", "-", t)         
    t = re.sub(r"[•·‣▪]", " ", t)                       
    t = re.sub(r"(?m)^\s*[-→]\s+", " ", t)               
    t = re.sub(r";", " ", t)                            
    t = re.sub(r"\s*([.,:!?=≠≥≤≈×±])\s*", r"\1", t)     
    t = re.sub(r"\s+", " ", t)                        
    return t.strip()

## Separate text from block elements

Defining helpers that separate plain text nuggets from [GRAFIK]/[FORMEL]/[CODE] blocks

In [ ]:
BLOCK_PREFIXES = ("[GRAFIK]", "[FORMEL]", "[CODE]")

def is_text_nugget(n: str) -> bool:
    return not n.lstrip().startswith(BLOCK_PREFIXES)

def text_only(page_content: str) -> str:
    content = page_content.replace("\\n", "\n")         
    blocks = re.split(r"\n\s*\n", content)
    return "\n\n".join(b for b in blocks if not b.lstrip().startswith(BLOCK_PREFIXES))

def build_parsetext(chunk: dict) -> str:
    parts = []
    if chunk.get("title"):
        parts.append("Titel: " + chunk["title"])
    parts.append(text_only(chunk.get("page_content", "")))
    return "\n".join(parts)

## Define Recall. Will later be aggregated

Defining recall_counts, the micro averaged exact recall of gold text facts against the parsed chunk

In [ ]:
def recall_counts(text_nuggets, chunk):
    if not text_nuggets:
        return 0, 0
    h = normalize(chunk)
    hits = sum(normalize(n) in h for n in text_nuggets)
    return hits, len(text_nuggets)

Configuring the LLM as judge client for the block modalities that need semantic matching

In [ ]:
load_dotenv(os.path.join(os.getcwd(), "..", ".env"), override=True)

GATEWAY_URL = os.getenv("GATEWAY_URL", "")
BEARER_TOKEN = os.getenv("BEARER_TOKEN", "")
JUDGE_MODEL = os.getenv("INFERENCE_MODEL_GATEWAY", "") 

client = OpenAI(base_url=GATEWAY_URL, api_key=BEARER_TOKEN)

print("Judge-LLM Model:", JUDGE_MODEL)

## LLM as a judge

Defining judge_item with caching, retries and JSON extraction, the semantic covered/missing decision per block element

In [ ]:
BLOCK_TYPES = ["grafik", "formel", "code"]

def build_fulltext(chunk):
    titel = "Titel: " + chunk["title"] + "\n" if chunk.get("title") else ""
    return titel + chunk.get("page_content", "")

cache = {}

def judge_item(element, parse_text, retries=2):
    key = (element, parse_text)
    if key in cache:
        return cache[key]

    prompt = f"""
Du bist ein Evaluator fuer Informations-Recall in Folien-Parsing.

Aufgabe:
Pruefe ob, die Information des gegebenen Elements irgendwo im geparsten Text enthalten ist.

Wichtig:
- Es spielt KEINE Rolle, wo die Information steht (Text, Grafikbeschreibung, Formel, Code).
- Es spielt KEINE Rolle, wie sie formuliert ist.
- Entscheidend ist nur semantische Ähnlichkeit.
- Zusaetzlicher Inhalt im Parse ist irrelevant.
- Du bewertest nur Recall: enthalten oder nicht enthalten.

Element:
{element}

Geparste Folie:
{parse_text}

Antworte strikt als JSON:
{{"verdict": "covered" oder "missing", "reason": "kurze Begründung"}}
"""

    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0,
            )
            parsed = json.loads(resp.choices[0].message.content)
            verdict = str(parsed.get("verdict", "")).strip().lower()
            if verdict not in ("covered", "missing"):
                raise ValueError(f"unerwartetes verdict: {verdict!r}")
            result = {"verdict": verdict, "reason": parsed.get("reason", "")}
            cache[key] = result
            return result
        except Exception as e:
            print(f"Try {attempt + 1} failed: {e}")

    return {"verdict": "invalid", "reason": f"Judge invalid after {retries} tries"}


In [ ]:
golden, by_id, pairs = load_pairs(CONFIGS[0])

# Pick the first block element (grafik/formel/code) we can find anywhere in the golden set and use it as a sanity probe for the judge.
test = None
for slide in golden:
    for block_type in BLOCK_TYPES:
        elements = slide.get(block_type, [])
        if elements:
            test = elements[0]
            break
    if test is not None:
        break

contains_text  = "Titel: Testfolie\n" + test
unrelated_text = "Titel: Organisatorisches\nDie Klausur findet am 15. Maerz statt; bitte rechtzeitig anmelden."

print("ELEMENT:\n", test, "\n")
print("contains it ->", judge_item(test, contains_text))
print("unrelated ->", judge_item(test, unrelated_text))

## Overall table: recall per modality (lectures pooled)

Pooling text/formel/code/grafik recall across both lectures into one overview table and saving parsing_gesamt.csv, the final parsing quality figure

In [ ]:

modality_counts = {}

def add_counts(modality, covered, total):
    if modality not in modality_counts:
        modality_counts[modality] = {"covered": 0, "total": 0}
    modality_counts[modality]["covered"] += covered
    modality_counts[modality]["total"] += total

for cfg in CONFIGS:
    golden, by_id, pairs = load_pairs(cfg)

    for g, p in pairs:
        text_nuggets = [n for n in g["text"] if is_text_nugget(n)]
        hits, total = recall_counts(text_nuggets, build_parsetext(p))
        add_counts("Text", hits, total)

    for g in golden:
        if g["slide_id"] not in by_id:
            continue
        parse_text = build_fulltext(by_id[g["slide_id"]])
        for mod in BLOCK_TYPES:
            for element in g.get(mod, []):
                verdict = judge_item(element, parse_text)["verdict"]
                if verdict == "covered":
                    add_counts(mod.capitalize(), 1, 1)
                elif verdict == "missing":
                    add_counts(mod.capitalize(), 0, 1)

MODALITY_ORDER = ["Text", "Formel", "Code", "Grafik"]
rows = []
overall_covered = 0
overall_total = 0
for modality in MODALITY_ORDER:
    counts = modality_counts.get(modality, {"covered": 0, "total": 0})
    covered = counts["covered"]
    total = counts["total"]
    if total == 0:
        continue
    rows.append({"Modalität": modality, "n": total, "Recall": covered / total})
    overall_covered += covered
    overall_total += total

rows.append({
    "Modalität": "Gesamt",
    "n": overall_total,
    "Recall": overall_covered / overall_total,
})

overview_df = pd.DataFrame(rows)
overview_df.to_csv(EVAL_OUT / "parsing_full_testset_final.csv", index=False, encoding="utf-8")
print("gespeichert:", EVAL_OUT / "parsing_full_testset_final.csv")

overview_df.style.format({"n": "{:.0f}", "Recall": "{:.3f}"}).hide(axis="index")

In [ ]:
missing_rows = []

for cfg in CONFIGS:
    golden, by_id, pairs = load_pairs(cfg)

    for g, p in pairs:
        parsetext = normalize(build_parsetext(p))
        for nugget in g["text"]:
            if not is_text_nugget(nugget):
                continue
            if normalize(nugget) not in parsetext:
                missing_rows.append({
                    "vorlesung": cfg["vorlesung"],
                    "slide": g["slide_id"],
                    "typ": "text",
                    "element": nugget,
                    "reason": "Text-Nugget nicht als (normalisierter) Substring im Parse gefunden",
                })

    for g in golden:
        if g["slide_id"] not in by_id:
            continue
        parse_text = build_fulltext(by_id[g["slide_id"]])
        for mod in BLOCK_TYPES:
            for element in g.get(mod, []):
                result = judge_item(element, parse_text)
                if result["verdict"] == "missing":
                    missing_rows.append({
                        "vorlesung": cfg["vorlesung"],
                        "slide": g["slide_id"],
                        "typ": mod,
                        "element": element,
                        "reason": result["reason"],
                    })

missing_df = pd.DataFrame(missing_rows, columns=["vorlesung", "slide", "typ", "element", "reason"])
missing_path = EVAL_OUT / "parsing_testrun_missing.csv"
missing_df.to_csv(missing_path, index=False, encoding="utf-8")
print("missing elements:", len(missing_df), "| by type:", missing_df["typ"].value_counts().to_dict())
print("saved ->", missing_path.resolve())
missing_df